# Multi-Sensor NDWI Analysis: Ayapel, Colombia

## Sentinel-2 and Landsat 8 | 2016-2025

Annual NDWI composites (2016-2025) for Ayapel, Cordoba, from Sentinel-2 and
Landsat 8 Surface Reflectance, exported as 10-band GeoTIFF stacks in EPSG:9377.

Ayapel (Cordoba) was selected for its wetland system, suitable for evaluating
multi-temporal surface-water conditions with NDWI. Sentinel-2 and Landsat 8 were
processed independently with sensor-specific cloud masks, and both resulting
10-band stacks (2016-2025) were exported as GeoTIFF in EPSG:9377.

**Limitation:** Sentinel-2 Level-2A data is unavailable for 2016, so `NDWI_2016`
in the Sentinel-2 stack has no information.


In [1]:
import os

PROJ_PATH = "/opt/conda/envs/geoprocesamiento/share/proj"

os.environ["PROJ_DATA"] = PROJ_PATH
os.environ["PROJ_LIB"] = PROJ_PATH

print("PROJ_DATA:", os.environ["PROJ_DATA"])
print("PROJ_LIB:", os.environ["PROJ_LIB"])

PROJ_DATA: /opt/conda/envs/geoprocesamiento/share/proj
PROJ_LIB: /opt/conda/envs/geoprocesamiento/share/proj


In [2]:
import ee
import geemap
import geopandas as gpd

from pathlib import Path
from pyproj import CRS

In [3]:
# Authenticate and initialize Google Earth Engine

try:
    ee.Initialize()
    print("Google Earth Engine initialized successfully.")

except Exception:
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine authenticated and initialized successfully.")

Enter verification code:  4/1ATsMZqAUOPgmkBehNr9c6bjQklMT7HSwt1blTUtR_zaPdP8PSBBK-JB5tVY



Successfully saved authorization token.
Google Earth Engine authenticated and initialized successfully.


In [4]:
# Root folder inside Docker
root_folder = Path("/workspace/TALLER_1")

# Load Colombian municipalities and reproject to the official
# Colombian projected CRS
gdf = gpd.read_file(root_folder / "municipios_colombia.gpkg")
gdf_unico = gdf.to_crs(epsg=9377)
gdf_unico["AREA_KM2"] = gdf_unico.geometry.area / 1_000_000

print(f"Initial CRS: {gdf.crs}")
gdf_unico.head()

Initial CRS: EPSG:3116


,DPTO_CCDGO,MPIO_CCDGO,MPIO_CNMBR,MPIO_CDPMP,VERSION,AREA,LATITUD,LONGITUD,STCTNENCUE,STP3_1_SI,...,STP51_SECU,STP51_SUPE,STP51_POST,STP51_13_E,STP51_99_E,Shape_Leng,Shape_Area,Codigo_Mun,geometry,AREA_KM2
0,18,001,FLORENCIA,18001,2018,2.547638e+09,1.749139,-75.558239,71877.0,32.0,...,59610.0,21898.0,4592.0,5892.0,3799.0,2.942508,0.206928,18001,"MULTIPOLYGON (((4730856.146 1800689.038, 47308...",2546.958298
1,18,029,ALBANIA,18029,2018,4.141221e+08,1.227865,-75.882327,2825.0,24.0,...,1712.0,231.0,41.0,215.0,46.0,1.112829,0.033618,18029,"MULTIPOLYGON (((4677933.827 1709133.846, 46779...",414.100691
2,18,094,BELÉN DE LOS ANDAQUÍES,18094,2018,1.191619e+09,1.500923,-75.875645,4243.0,54.0,...,3340.0,490.0,119.0,720.0,123.0,2.234657,0.096745,18094,"MULTIPOLYGON (((4690015.614 1751610.86, 469000...",1191.551389
3,18,247,EL DONCELLO,18247,2018,1.106076e+09,1.791386,-75.193944,8809.0,0.0,...,6287.0,1029.0,228.0,1095.0,171.0,3.154370,0.089867,18247,"MULTIPOLYGON (((4737450.122 1814755.048, 47374...",1105.514836
4,18,256,EL PAUJÍL,18256,2018,1.234734e+09,1.617746,-75.234043,5795.0,0.0,...,4066.0,639.0,108.0,916.0,99.0,3.529316,0.100309,18256,"MULTIPOLYGON (((4736905.653 1802381.382, 47376...",1234.140645


In [5]:
# Select Ayapel municipality in Córdoba

gdf_muni = gdf_unico[
    (gdf_unico["MPIO_CNMBR"].str.upper() == "AYAPEL")
    & (gdf_unico["DPTO_CCDGO"].astype(str).str.zfill(2) == "23")
].copy()

if gdf_muni.empty:
    raise ValueError("Ayapel municipality was not found.")

gdf_muni[["DPTO_CCDGO", "MPIO_CNMBR", "AREA_KM2"]]

,DPTO_CCDGO,MPIO_CNMBR,AREA_KM2
437,23,AYAPEL,1963.614618


In [6]:
# Convert municipality geometry to WGS84
gdf_muni_wgs84 = gdf_muni.to_crs(epsg=4326)
geojson_geom = gdf_muni_wgs84.geometry.iloc[0].__geo_interface__
ee_muni_geom = ee.Geometry(geojson_geom)

print("Ayapel geometry successfully converted to Earth Engine.")

Ayapel geometry successfully converted to Earth Engine.


In [7]:
# Calculate centroid in projected CRS
centroid_point = gdf_muni.geometry.centroid.to_crs(epsg=4326).iloc[0]
centroid = [centroid_point.y, centroid_point.x]

print("Map centroid:", centroid)

Map centroid: [8.265753747003297, -75.04871885063757]


In [8]:
Map_AOI = geemap.Map(center=centroid, zoom=10)
Map_AOI.add_gdf(gdf_muni_wgs84, layer_name="Ayapel municipality")
Map_AOI

Map(center=[8.265753747003297, -75.04871885063757], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
def mask_scale_s2(image):
    """Mask clouds/invalid pixels with the Sentinel-2 Scene Classification
    Layer (SCL) and scale B3/B8 to Surface Reflectance."""

    scl = image.select("SCL")

    mask = (
        scl.neq(0)        # No data
        .And(scl.neq(1))  # Saturated / defective
        .And(scl.neq(3))  # Cloud shadow
        .And(scl.neq(8))  # Medium probability cloud
        .And(scl.neq(9))  # High probability cloud
        .And(scl.neq(10)) # Cirrus
        .And(scl.neq(11)) # Snow / ice
    )

    reflectance = image.select(["B3", "B8"]).multiply(0.0001).toFloat()

    return reflectance.updateMask(mask).copyProperties(
        image, ["system:time_start"]
    )


def mask_scale_l8(image):
    """Mask clouds/invalid pixels with QA_PIXEL/QA_RADSAT and apply the
    Landsat 8 Collection 2 Surface Reflectance scaling factors."""

    qa = image.select("QA_PIXEL")
    # Bits 0-5: fill, dilated cloud, cirrus, cloud, cloud shadow, snow
    qa_mask = qa.bitwiseAnd(0b111111).eq(0)
    saturation_mask = image.select("QA_RADSAT").eq(0)

    reflectance = (
        image.select(["SR_B3", "SR_B5"]).multiply(0.0000275).add(-0.2).toFloat()
    )

    return (
        reflectance.updateMask(qa_mask)
        .updateMask(saturation_mask)
        .copyProperties(image, ["system:time_start"])
    )

In [10]:
def calculate_ndwi(image, green_band, nir_band):
    """Generic NDWI calculation, reused for any sensor: (Green - NIR) / (Green + NIR)."""

    green = image.select(green_band)
    nir = image.select(nir_band)
    denominator = green.add(nir)

    return (
        green.subtract(nir)
        .divide(denominator)
        .updateMask(denominator.neq(0))
        .rename("NDWI")
    )

In [11]:
INDEX_NAME = "NDWI"

# One config per sensor instead of duplicated annual_s2_* / annual_l8_*
# functions. Add a new sensor by adding a new entry here.
SENSOR_CONFIGS = {
    "S2": {
        "collection_id": "COPERNICUS/S2_SR_HARMONIZED",
        "green_band": "B3",
        "nir_band": "B8",
        "mask_fn": mask_scale_s2,
        "cloud_field": "CLOUDY_PIXEL_PERCENTAGE",
        "cloud_thresh": 30,
        "scale": 10,
        "min_year": 2017,  # S2 SR is not available for 2016
    },
    "L8": {
        "collection_id": "LANDSAT/LC08/C02/T1_L2",
        "green_band": "SR_B3",
        "nir_band": "SR_B5",
        "mask_fn": mask_scale_l8,
        "cloud_field": "CLOUD_COVER",
        "cloud_thresh": 30,
        "scale": 30,
        "min_year": 2016,
    },
}


def annual_index(year, cfg, aoi):
    """Build one annual median-composite index image for a given sensor config."""

    band_name = f"{INDEX_NAME}_{year}"

    if year < cfg["min_year"]:
        empty_image = (
            ee.Image.constant(0)
            .toFloat()
            .rename(band_name)
            .updateMask(ee.Image.constant(0))
            .clip(aoi)
        )
        return empty_image.set(
            {"year": year, "scene_count": 0, "status": "No data available"}
        )

    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, "year")  # filterDate() end is exclusive

    collection = (
        ee.ImageCollection(cfg["collection_id"])
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt(cfg["cloud_field"], cfg["cloud_thresh"]))
        .map(cfg["mask_fn"])
    )

    composite = collection.median()
    index_image = (
        calculate_ndwi(composite, cfg["green_band"], cfg["nir_band"])
        .rename(band_name)
        .clip(aoi)
    )

    return index_image.set({"year": year, "scene_count": collection.size()})

In [12]:
years = list(range(2016, 2026))

annual_images = {
    sensor: [annual_index(year, cfg, ee_muni_geom) for year in years]
    for sensor, cfg in SENSOR_CONFIGS.items()
}

print("Annual image objects successfully created for:", list(annual_images.keys()))

Annual image objects successfully created for: ['S2', 'L8']


In [13]:
# Single getInfo() call for all sensors/years instead of one call per
# year-sensor combination (20 round-trips in the original version).
scene_counts = ee.Dictionary(
    {
        sensor: ee.List(images).map(lambda img: ee.Image(img).get("scene_count"))
        for sensor, images in annual_images.items()
    }
).getInfo()

for sensor, counts in scene_counts.items():
    print(sensor)
    print("-" * 30)
    for year, count in zip(years, counts):
        print(f"{year}: {count} scenes")
    print()

L8
------------------------------
2016: 9 scenes
2017: 11 scenes
2018: 8 scenes
2019: 8 scenes
2020: 7 scenes
2021: 7 scenes
2022: 6 scenes
2023: 9 scenes
2024: 5 scenes
2025: 7 scenes

S2
------------------------------
2016: 0 scenes
2017: 4 scenes
2018: 19 scenes
2019: 82 scenes
2020: 93 scenes
2021: 69 scenes
2022: 39 scenes
2023: 34 scenes
2024: 44 scenes
2025: 29 scenes



In [14]:
def build_stack(images):
    """Combine a list of single-band annual images into one multi-band ee.Image."""

    stack = ee.Image(images[0])
    for image in images[1:]:
        stack = stack.addBands(image)
    return stack.toFloat()


stacks = {sensor: build_stack(images) for sensor, images in annual_images.items()}

s2_ndwi_stack = stacks["S2"]
l8_ndwi_stack = stacks["L8"]

print("Sentinel-2 and Landsat 8 stacks created.")

Sentinel-2 and Landsat 8 stacks created.


In [15]:
for sensor, stack in stacks.items():
    print(f"{sensor} bands:", stack.bandNames().getInfo())

S2 bands: ['NDWI_2016', 'NDWI_2017', 'NDWI_2018', 'NDWI_2019', 'NDWI_2020', 'NDWI_2021', 'NDWI_2022', 'NDWI_2023', 'NDWI_2024', 'NDWI_2025']
L8 bands: ['NDWI_2016', 'NDWI_2017', 'NDWI_2018', 'NDWI_2019', 'NDWI_2020', 'NDWI_2021', 'NDWI_2022', 'NDWI_2023', 'NDWI_2024', 'NDWI_2025']


In [26]:
import ipywidgets as widgets
from IPython.display import display

ndwi_vis = {
    "min": -0.5,
    "max": 0.8,
    "palette": ["brown", "yellow", "white", "lightblue", "blue", "darkblue"],
}

Map_NDWI_years = geemap.Map(center=centroid, zoom=10)
Map_NDWI_years.addLayer(ee_muni_geom, {"color": "black"}, "Ayapel boundary")

sensor_dropdown = widgets.Dropdown(options=list(stacks.keys()), value="S2", description="Sensor:")
year_dropdown = widgets.Dropdown(options=years, value=years[-1], description="Year:")


def update_layer(change=None):


    sensor = sensor_dropdown.value
    year = year_dropdown.value
    band_name = f"{INDEX_NAME}_{year}"

    # Remove the previous NDWI layer (if any) before adding the new one
    existing_layer = Map_NDWI_years.find_layer("NDWI")
    if existing_layer is not None:
        Map_NDWI_years.remove_layer(existing_layer)

    Map_NDWI_years.addLayer(
        stacks[sensor].select(band_name), ndwi_vis, "NDWI"
    )


sensor_dropdown.observe(update_layer, names="value")
year_dropdown.observe(update_layer, names="value")

update_layer()  # draw the initial layer (S2, last year)

display(widgets.HBox([sensor_dropdown, year_dropdown]))
Map_NDWI_years

Map(center=[8.265753747003297, -75.04871885063757], controls=(WidgetControl(options=['position', 'transparent_…

In [17]:
# Build EPSG:9377 as WKT because Earth Engine does not parse the
# EPSG:9377 authority code directly
crs_9377_wkt = CRS.from_epsg(9377).to_wkt(version="WKT1_GDAL")
ee.Projection(crs_9377_wkt).getInfo()

{'type': 'Projection',
 'wkt': 'PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional", \n  GEOGCS["MAGNA-SIRGAS 2018", \n    DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018", \n      SPHEROID["GRS 1980", 6378137.0, 298.257222101, AUTHORITY["EPSG","7019"]], \n      AUTHORITY["EPSG","1329"]], \n    PRIMEM["Greenwich", 0.0, AUTHORITY["EPSG","8901"]], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH], \n    AUTHORITY["EPSG","20046"]], \n  PROJECTION["Transverse_Mercator"], \n  PARAMETER["central_meridian", -73.0], \n  PARAMETER["latitude_of_origin", 4.0], \n  PARAMETER["scale_factor", 0.9992], \n  PARAMETER["false_easting", 5000000.0], \n  PARAMETER["false_northing", 2000000.0], \n  UNIT["m", 1.0], \n  AXIS["x", EAST], \n  AXIS["y", NORTH], \n  AUTHORITY["EPSG","9377"]]',
 'transform': [1, 0, 0, 0, 1, 0]}

In [18]:
export_tasks = {}

for sensor, stack in stacks.items():
    description = f"{sensor}_NDWI_Ayapel_2016_2025"
    task = ee.batch.Export.image.toDrive(
        image=stack,
        description=description,
        folder="GEE_EXPORTS_GEOPRO",
        fileNamePrefix=description,
        region=ee_muni_geom,
        crs=crs_9377_wkt,
        scale=SENSOR_CONFIGS[sensor]["scale"],
        maxPixels=1e13,
        fileFormat="GeoTIFF",
    )
    task.start()
    export_tasks[sensor] = task
    print(f"{description} export started.")

S2_NDWI_Ayapel_2016_2025 export started.
L8_NDWI_Ayapel_2016_2025 export started.


In [19]:
for task in ee.batch.Task.list():
    status = task.status()
    print("=" * 60)
    print("Task:", status.get("description"))
    print("State:", status.get("state"))
    print("Error:", status.get("error_message", "No error message"))

Task: L8_NDWI_Ayapel_2016_2025
State: READY
Error: No error message
Task: S2_NDWI_Ayapel_2016_2025
State: READY
Error: No error message
Task: L8_NDWI_Ayapel_2016_2025
State: COMPLETED
Error: No error message
Task: S2_NDWI_Ayapel_2016_2025
State: COMPLETED
Error: No error message
Task: L8_NDWI_Ayapel_2016_2025
State: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: S2_NDWI_Ayapel_2016_2025
State: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: L8_NDWI_Ayapel_2016_2025
State: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: S2_NDWI_Ayapel_2016_2025
State: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: S2_NDWI_Ayapel_2016_2025
State: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: L8_NDWI_Ayapel_2016_2025
State: FAILED
Error: Projection: The CRS of a map projection could not be parsed.
Task: S2_NDWI_Ayapel_2016_2025
State: FAIL

In [20]:
from pyproj import CRS

# Build EPSG:9377 as WKT because Earth Engine
# does not parse the EPSG:9377 authority code directly
crs_9377_wkt = CRS.from_epsg(9377).to_wkt(
    version="WKT1_GDAL"
)

print(crs_9377_wkt)

PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional",GEOGCS["MAGNA-SIRGAS 2018",DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1329"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","20046"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",4],PARAMETER["central_meridian",-73],PARAMETER["scale_factor",0.9992],PARAMETER["false_easting",5000000],PARAMETER["false_northing",2000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AUTHORITY["EPSG","9377"]]


In [21]:
test_projection = ee.Projection(crs_9377_wkt)

print(test_projection.getInfo())

{'type': 'Projection', 'wkt': 'PROJCS["MAGNA-SIRGAS 2018 / Origen-Nacional", \n  GEOGCS["MAGNA-SIRGAS 2018", \n    DATUM["Marco_Geocentrico_Nacional_de_Referencia_2018", \n      SPHEROID["GRS 1980", 6378137.0, 298.257222101, AUTHORITY["EPSG","7019"]], \n      AUTHORITY["EPSG","1329"]], \n    PRIMEM["Greenwich", 0.0, AUTHORITY["EPSG","8901"]], \n    UNIT["degree", 0.017453292519943295], \n    AXIS["Longitude", EAST], \n    AXIS["Latitude", NORTH], \n    AUTHORITY["EPSG","20046"]], \n  PROJECTION["Transverse_Mercator"], \n  PARAMETER["central_meridian", -73.0], \n  PARAMETER["latitude_of_origin", 4.0], \n  PARAMETER["scale_factor", 0.9992], \n  PARAMETER["false_easting", 5000000.0], \n  PARAMETER["false_northing", 2000000.0], \n  UNIT["m", 1.0], \n  AXIS["x", EAST], \n  AXIS["y", NORTH], \n  AUTHORITY["EPSG","9377"]]', 'transform': [1, 0, 0, 0, 1, 0]}


In [22]:
s2_export = ee.batch.Export.image.toDrive(
    image=s2_ndwi_stack,
    description="S2_NDWI_Ayapel_2016_2025",
    folder="GEE_EXPORTS_GEOPRO",
    fileNamePrefix="S2_NDWI_Ayapel_2016_2025",
    region=ee_muni_geom,
    crs=crs_9377_wkt,
    scale=10,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

s2_export.start()

print("Sentinel-2 NDWI export iniciada.")

Sentinel-2 NDWI export iniciada.


In [23]:
l8_export = ee.batch.Export.image.toDrive(
    image=l8_ndwi_stack,
    description="L8_NDWI_Ayapel_2016_2025",
    folder="GEE_EXPORTS_GEOPRO",
    fileNamePrefix="L8_NDWI_Ayapel_2016_2025",
    region=ee_muni_geom,
    crs=crs_9377_wkt,
    scale=30,
    maxPixels=1e13,
    fileFormat="GeoTIFF"
)

l8_export.start()

print("Landsat 8 NDWI export iniciada.")

Landsat 8 NDWI export iniciada.


## Results download

The exported stacks (10 bands, EPSG:9377, GeoTIFF) are available for download from Google Drive:

- **Landsat 8** (`L8_NDWI_Ayapel_2016_2025.tif`): [Download](https://drive.google.com/file/d/1V2r9KveH-d9UEGniNVT7NWaxvicK9YMK/view?usp=drive_link)
- **Sentinel-2** (`S2_NDWI_Ayapel_2016_2025.tif`): [Download](https://drive.google.com/file/d/15H8FWJG04SSIF5ONoA1XzDJzcfsR3VpV/view?usp=drive_link)